<a href="https://colab.research.google.com/github/Qalani/Dissertation/blob/main/Winam_Manual_Review_LowConfidence_Masks_With_Date_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Winam Gulf manual-review mask workflow

This notebook creates **date-filtered low-confidence review masks** from the output of `Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb`.

It is designed for the following loop:

1. Read the classifier batch run log.
2. Filter scenes by date and/or sensor.
3. Create GeoTIFF masks for:
   - low-confidence pixels,
   - low-confidence floating-plant pixels,
   - model–rule disagreement pixels,
   - low-confidence + model–rule disagreement pixels.
4. Write a review manifest for QGIS.
5. Optionally sample manually corrected QGIS points back into new training CSV rows.

The probability rasters created by the classifier notebook are assumed to be **maximum predicted class probability**, scaled `0–100`, with `255` as nodata.


## 1. Mount Google Drive

Run this in Colab before using Drive paths.


In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
else:
    print('Not running in Colab; skipping Drive mount.')


Mounted at /content/drive


## 2. Imports

If `rasterio` or `geopandas` are missing in a fresh Colab runtime, uncomment the install line below and re-run the cell.


In [ ]:
# Uncomment in a fresh Colab runtime if imports fail:
# !pip -q install rasterio geopandas pyogrio shapely fiona

import ast
import json
import math
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.errors import RasterioIOError

try:
    import geopandas as gpd
    from shapely.geometry import Point
    GEOPANDAS_AVAILABLE = True
except Exception as exc:
    GEOPANDAS_AVAILABLE = False
    print('geopandas not available. Mask generation will still work; optional point sampling will not.')
    print(type(exc).__name__, exc)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)


## 3. Configuration

Edit the date filter here.

Date-filter modes:

- `overlap`: keep scenes whose date window overlaps your filter window.
- `start_within`: keep scenes whose `start_date` falls inside the filter window.
- `contained`: keep scenes fully contained within the filter window.

For all dates, set both `REVIEW_START_DATE` and `REVIEW_END_DATE` to `None`.


In [ ]:
# ---------------------------------------------------------------------
# Drive/output paths
# ---------------------------------------------------------------------
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
CLASSIFIER_ROOT = DRIVE_MYDRIVE / 'Winam_RF_Training_Data'
CLASSIFIER_BATCH_DIR = CLASSIFIER_ROOT / 'outputs' / 'full_stack_batch'
CLASSIFIER_TABLE_DIR = CLASSIFIER_BATCH_DIR / 'tables'

# Leave as None to auto-detect the latest run-log CSV in CLASSIFIER_TABLE_DIR.
RUN_LOG_PATH = None

# Manual-review outputs will be written here.
REVIEW_DIR = CLASSIFIER_ROOT / 'outputs' / 'manual_review'
MASK_DIR = REVIEW_DIR / 'masks'
TABLE_DIR = REVIEW_DIR / 'tables'
TEMPLATE_DIR = REVIEW_DIR / 'templates'

# ---------------------------------------------------------------------
# Date filtering
# ---------------------------------------------------------------------
REVIEW_START_DATE = None        # e.g. '2021-01-01'
REVIEW_END_DATE = None          # e.g. '2021-12-31'
DATE_FILTER_MODE = 'overlap'    # 'overlap', 'start_within', or 'contained'

# Sensor filtering. Use None for both sensors, or e.g. ['S2'] / ['S1'].
SENSOR_FILTER = None            # e.g. ['S2']

# Optional prefix filtering. Useful for a single scene/date-window.
PREFIX_CONTAINS = None          # e.g. '2021-02-01_to_2021-03-01'

# ---------------------------------------------------------------------
# Mask settings
# ---------------------------------------------------------------------
NODATA_VALUE = 255
LOW_PROBA_THRESHOLD = 30        # probability raster scale is 0..100
FLOATING_CLASS_CODE = 2
OVERWRITE_OUTPUTS = False

# Create these outputs. Leave all True unless you only need one kind of mask.
WRITE_LOW_CONFIDENCE_ALL = True
WRITE_LOW_CONFIDENCE_FLOATING = True
WRITE_RULE_DISAGREEMENT = True
WRITE_LOW_CONFIDENCE_AND_DISAGREEMENT = True

# Local staging is safer in Colab because raster block I/O directly on Google Drive can be unstable.
USE_LOCAL_STAGING = True
LOCAL_STAGE_DIR = Path('/content/winam_manual_review_stage')

for p in [REVIEW_DIR, MASK_DIR, TABLE_DIR, TEMPLATE_DIR, LOCAL_STAGE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Manual-review output directory:', REVIEW_DIR)


Manual-review output directory: /content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review


## 4. Load and filter the classifier run log

This cell auto-detects the most recently modified `winam_full_stack_run_log*.csv` unless you specify `RUN_LOG_PATH` above.


In [ ]:
def auto_find_latest_run_log(table_dir):
    table_dir = Path(table_dir)
    candidates = sorted(table_dir.glob('winam_full_stack_run_log*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No winam_full_stack_run_log*.csv found in {table_dir}')
    return candidates[0]


def coerce_date_series(s):
    return pd.to_datetime(s, errors='coerce').dt.tz_localize(None)


def filter_by_date_window(df, start_date=None, end_date=None, mode='overlap'):
    df = df.copy()
    df['_start_dt'] = coerce_date_series(df['start_date'])
    df['_end_dt'] = coerce_date_series(df['end_date'])

    # If end_date is missing in the run log, treat it as the same as start_date.
    df['_end_dt'] = df['_end_dt'].fillna(df['_start_dt'])

    start = pd.to_datetime(start_date) if start_date else None
    end = pd.to_datetime(end_date) if end_date else None

    mask = pd.Series(True, index=df.index)

    if start is not None and end is not None and start > end:
        raise ValueError('REVIEW_START_DATE is after REVIEW_END_DATE.')

    if start is not None or end is not None:
        if mode == 'overlap':
            # keep scenes where scene_end >= filter_start AND scene_start <= filter_end
            if start is not None:
                mask &= df['_end_dt'] >= start
            if end is not None:
                mask &= df['_start_dt'] <= end
        elif mode == 'start_within':
            if start is not None:
                mask &= df['_start_dt'] >= start
            if end is not None:
                mask &= df['_start_dt'] <= end
        elif mode == 'contained':
            if start is not None:
                mask &= df['_start_dt'] >= start
            if end is not None:
                mask &= df['_end_dt'] <= end
        else:
            raise ValueError("DATE_FILTER_MODE must be 'overlap', 'start_within', or 'contained'.")

    return df[mask].drop(columns=['_start_dt', '_end_dt'])


run_log_path = Path(RUN_LOG_PATH) if RUN_LOG_PATH else auto_find_latest_run_log(CLASSIFIER_TABLE_DIR)
print('Using run log:', run_log_path)

run_log = pd.read_csv(run_log_path)
print('Run-log rows:', len(run_log))
print('Run-log columns:')
print(list(run_log.columns))

required_cols = ['sensor', 'start_date', 'end_date', 'prefix', 'status', 'model_classification_tif', 'model_probability_tif']
missing = [c for c in required_cols if c not in run_log.columns]
if missing:
    raise ValueError(f'Run log is missing required columns: {missing}')

completed = run_log[
    run_log['status'].astype(str).str.lower().eq('completed') &
    run_log['model_classification_tif'].notna() &
    run_log['model_probability_tif'].notna()
].copy()

if SENSOR_FILTER is not None:
    completed = completed[completed['sensor'].isin(SENSOR_FILTER)].copy()

if PREFIX_CONTAINS:
    completed = completed[completed['prefix'].astype(str).str.contains(PREFIX_CONTAINS, regex=False, na=False)].copy()

selected = filter_by_date_window(
    completed,
    start_date=REVIEW_START_DATE,
    end_date=REVIEW_END_DATE,
    mode=DATE_FILTER_MODE,
).reset_index(drop=True)

print(f'Completed rows after sensor/prefix/date filtering: {len(selected)}')
if selected.empty:
    raise RuntimeError('No scenes matched the filter. Check dates, sensor filter, prefix filter, and run-log path.')

preview_cols = ['sensor', 'start_date', 'end_date', 'prefix', 'model_classification_tif', 'model_probability_tif']
if 'rule_classification_tif' in selected.columns:
    preview_cols.append('rule_classification_tif')
display(selected[preview_cols].head(20))


Using run log: /content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/tables/winam_full_stack_run_log_route_b_s1_scc_spatialcv_proba_v3.csv
Run-log rows: 814
Run-log columns:
['sensor', 'start_date', 'end_date', 'prefix', 'status', 'n_predictor_tifs', 'predictor_tifs', 'model_classification_tif', 'model_probability_tif', 'rule_classification_tif', 'area_csv', 'confidence_csv', 'quicklook_pngs', 'error_type', 'error_message']
Completed rows after sensor/prefix/date filtering: 100


,sensor,start_date,end_date,prefix,model_classification_tif,model_probability_tif,rule_classification_tif
0,S2,2017-05-22,2017-05-23,winam_s2_predictors_2017-05-22_to_2017-05-23,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...
1,S1,2025-08-30,2025-08-31,winam_s1_scc_predictors_2025-08-30_to_2025-08-31,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...
2,S2,2025-08-30,2025-08-31,winam_s2_predictors_2025-08-30_to_2025-08-31,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...
3,S2,2025-09-02,2025-09-03,winam_s2_predictors_2025-09-02_to_2025-09-03,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-02_to_2025-09-03_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-02_to_2025-09-03_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-02_to_2025-09-03_...
4,S1,2025-09-03,2025-09-04,winam_s1_scc_predictors_2025-09-03_to_2025-09-04,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-03_to_2025-09...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-03_to_2025-09...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-03_to_2025-09...
5,S2,2025-09-09,2025-09-10,winam_s2_predictors_2025-09-09_to_2025-09-10,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-09_to_2025-09-10_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-09_to_2025-09-10_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-09_to_2025-09-10_...
6,S1,2025-09-11,2025-09-12,winam_s1_scc_predictors_2025-09-11_to_2025-09-12,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-11_to_2025-09...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-11_to_2025-09...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-09-11_to_2025-09...
7,S2,2025-09-12,2025-09-13,winam_s2_predictors_2025-09-12_to_2025-09-13,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-12_to_2025-09-13_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-12_to_2025-09-13_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geot

## 5. Helper functions for safe raster I/O and mask writing


In [ ]:
def path_exists_nonempty(path_value):
    if path_value is None or (isinstance(path_value, float) and math.isnan(path_value)):
        return False
    path = Path(str(path_value))
    return path.exists() and path.stat().st_size > 0


def stage_for_read(path_value, stage_dir=LOCAL_STAGE_DIR, use_local=USE_LOCAL_STAGING):
    """Copy a Drive raster to local /content before block I/O; return the local or original path."""
    source = Path(str(path_value))
    if not source.exists():
        raise FileNotFoundError(source)
    if not use_local:
        return source

    stage_dir = Path(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)
    local = stage_dir / source.name

    if local.exists() and local.stat().st_size == source.stat().st_size:
        return local

    print(f'Staging for read: {source.name}')
    shutil.copy2(source, local)
    return local


def final_output_path(prefix, suffix):
    safe_prefix = str(prefix).replace('/', '_').replace(' ', '_')
    return MASK_DIR / f'{safe_prefix}_{suffix}.tif'


def local_output_path(final_path):
    final_path = Path(final_path)
    LOCAL_STAGE_DIR.mkdir(parents=True, exist_ok=True)
    return LOCAL_STAGE_DIR / final_path.name


def copy_local_to_final(local_path, final_path):
    final_path = Path(final_path)
    final_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, final_path)
    return final_path


def write_mask_from_class_probability(
    class_tif,
    proba_tif,
    output_tif,
    threshold=70,
    floating_only=False,
    nodata_value=255,
    floating_class_code=2,
    overwrite=False,
):
    """Write uint8 mask: 1 = review target, 0 = all other pixels."""
    output_tif = Path(output_tif)
    if output_tif.exists() and not overwrite:
        return output_tif, 'exists_skipped'

    class_local = stage_for_read(class_tif)
    proba_local = stage_for_read(proba_tif)
    temp_out = local_output_path(output_tif) if USE_LOCAL_STAGING else output_tif

    with rasterio.open(class_local) as cls_src, rasterio.open(proba_local) as proba_src:
        if cls_src.shape != proba_src.shape:
            raise ValueError(f'Shape mismatch: {Path(class_tif).name} and {Path(proba_tif).name}')
        profile = cls_src.profile.copy()
        profile.update(count=1, dtype='uint8', nodata=0, compress='lzw', BIGTIFF='YES')

        with rasterio.open(temp_out, 'w', **profile) as dst:
            for _, window in cls_src.block_windows(1):
                cls = cls_src.read(1, window=window)
                proba = proba_src.read(1, window=window)
                valid = (cls != nodata_value) & (proba != nodata_value)
                mask = valid & (proba < int(threshold))
                if floating_only:
                    mask &= cls == floating_class_code
                dst.write(mask.astype('uint8'), 1, window=window)

    if USE_LOCAL_STAGING:
        copy_local_to_final(temp_out, output_tif)
    return output_tif, 'written'


def write_rule_disagreement_mask(class_tif, rule_tif, output_tif, nodata_value=255, overwrite=False):
    """Write uint8 mask: 1 = model class differs from rule class, 0 = agreement/invalid."""
    output_tif = Path(output_tif)
    if output_tif.exists() and not overwrite:
        return output_tif, 'exists_skipped'

    class_local = stage_for_read(class_tif)
    rule_local = stage_for_read(rule_tif)
    temp_out = local_output_path(output_tif) if USE_LOCAL_STAGING else output_tif

    with rasterio.open(class_local) as cls_src, rasterio.open(rule_local) as rule_src:
        if cls_src.shape != rule_src.shape:
            raise ValueError(f'Shape mismatch: {Path(class_tif).name} and {Path(rule_tif).name}')
        profile = cls_src.profile.copy()
        profile.update(count=1, dtype='uint8', nodata=0, compress='lzw', BIGTIFF='YES')

        with rasterio.open(temp_out, 'w', **profile) as dst:
            for _, window in cls_src.block_windows(1):
                cls = cls_src.read(1, window=window)
                rule = rule_src.read(1, window=window)
                valid = (cls != nodata_value) & (rule != nodata_value)
                mask = valid & (cls != rule)
                dst.write(mask.astype('uint8'), 1, window=window)

    if USE_LOCAL_STAGING:
        copy_local_to_final(temp_out, output_tif)
    return output_tif, 'written'


def write_lowconf_disagreement_mask(class_tif, proba_tif, rule_tif, output_tif, threshold=70, nodata_value=255, overwrite=False):
    """Write uint8 mask: 1 = low probability and model-rule disagreement."""
    output_tif = Path(output_tif)
    if output_tif.exists() and not overwrite:
        return output_tif, 'exists_skipped'

    class_local = stage_for_read(class_tif)
    proba_local = stage_for_read(proba_tif)
    rule_local = stage_for_read(rule_tif)
    temp_out = local_output_path(output_tif) if USE_LOCAL_STAGING else output_tif

    with rasterio.open(class_local) as cls_src, rasterio.open(proba_local) as proba_src, rasterio.open(rule_local) as rule_src:
        if cls_src.shape != proba_src.shape or cls_src.shape != rule_src.shape:
            raise ValueError(f'Shape mismatch among class/probability/rule rasters for {Path(class_tif).name}')
        profile = cls_src.profile.copy()
        profile.update(count=1, dtype='uint8', nodata=0, compress='lzw', BIGTIFF='YES')

        with rasterio.open(temp_out, 'w', **profile) as dst:
            for _, window in cls_src.block_windows(1):
                cls = cls_src.read(1, window=window)
                proba = proba_src.read(1, window=window)
                rule = rule_src.read(1, window=window)
                valid = (cls != nodata_value) & (proba != nodata_value) & (rule != nodata_value)
                mask = valid & (proba < int(threshold)) & (cls != rule)
                dst.write(mask.astype('uint8'), 1, window=window)

    if USE_LOCAL_STAGING:
        copy_local_to_final(temp_out, output_tif)
    return output_tif, 'written'


def count_mask_pixels(mask_tif):
    """Return the number of pixels equal to 1 in a uint8 mask."""
    mask_tif = Path(mask_tif)
    total = 0
    if not mask_tif.exists():
        return np.nan
    with rasterio.open(mask_tif) as src:
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)
            total += int((arr == 1).sum())
    return total


## 6. Create date-filtered review masks

This writes one or more GeoTIFF masks per selected scene into:

```text
/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/
```

Each mask uses:

- `1` = pixel selected for review
- `0` = not selected / invalid


In [ ]:
review_rows = []
errors = []

for i, row in selected.iterrows():
    prefix = row['prefix']
    sensor = row['sensor']
    class_tif = row['model_classification_tif']
    proba_tif = row['model_probability_tif']
    rule_tif = row.get('rule_classification_tif', '')

    print(f"\n[{i+1}/{len(selected)}] {sensor} {row['start_date']} to {row['end_date']} | {prefix}")

    base = {
        'sensor': sensor,
        'start_date': row['start_date'],
        'end_date': row['end_date'],
        'prefix': prefix,
        'class_tif': class_tif,
        'proba_tif': proba_tif,
        'rule_tif': rule_tif,
        'low_proba_threshold': LOW_PROBA_THRESHOLD,
    }

    if not path_exists_nonempty(class_tif) or not path_exists_nonempty(proba_tif):
        msg = 'Missing class or probability raster.'
        print('  SKIP:', msg)
        errors.append({**base, 'error': msg})
        continue

    outputs = {}
    statuses = {}

    try:
        if WRITE_LOW_CONFIDENCE_ALL:
            out = final_output_path(prefix, f'lowconf_lt{LOW_PROBA_THRESHOLD}')
            path, status = write_mask_from_class_probability(
                class_tif, proba_tif, out,
                threshold=LOW_PROBA_THRESHOLD,
                floating_only=False,
                nodata_value=NODATA_VALUE,
                floating_class_code=FLOATING_CLASS_CODE,
                overwrite=OVERWRITE_OUTPUTS,
            )
            outputs['lowconf_mask'] = str(path)
            statuses['lowconf_mask_status'] = status

        if WRITE_LOW_CONFIDENCE_FLOATING:
            out = final_output_path(prefix, f'floating_lowconf_lt{LOW_PROBA_THRESHOLD}')
            path, status = write_mask_from_class_probability(
                class_tif, proba_tif, out,
                threshold=LOW_PROBA_THRESHOLD,
                floating_only=True,
                nodata_value=NODATA_VALUE,
                floating_class_code=FLOATING_CLASS_CODE,
                overwrite=OVERWRITE_OUTPUTS,
            )
            outputs['floating_lowconf_mask'] = str(path)
            statuses['floating_lowconf_mask_status'] = status

        has_rule = path_exists_nonempty(rule_tif)
        if WRITE_RULE_DISAGREEMENT and has_rule:
            out = final_output_path(prefix, 'model_rule_disagreement')
            path, status = write_rule_disagreement_mask(
                class_tif, rule_tif, out,
                nodata_value=NODATA_VALUE,
                overwrite=OVERWRITE_OUTPUTS,
            )
            outputs['model_rule_disagreement_mask'] = str(path)
            statuses['model_rule_disagreement_mask_status'] = status

        if WRITE_LOW_CONFIDENCE_AND_DISAGREEMENT and has_rule:
            out = final_output_path(prefix, f'lowconf_lt{LOW_PROBA_THRESHOLD}_and_model_rule_disagreement')
            path, status = write_lowconf_disagreement_mask(
                class_tif, proba_tif, rule_tif, out,
                threshold=LOW_PROBA_THRESHOLD,
                nodata_value=NODATA_VALUE,
                overwrite=OVERWRITE_OUTPUTS,
            )
            outputs['lowconf_disagreement_mask'] = str(path)
            statuses['lowconf_disagreement_mask_status'] = status

        if not has_rule:
            outputs['model_rule_disagreement_mask'] = ''
            outputs['lowconf_disagreement_mask'] = ''
            statuses['rule_related_status'] = 'no_rule_raster'

        # Pixel counts are useful for prioritising QGIS review.
        for key, path in list(outputs.items()):
            if path:
                outputs[f'{key}_pixel_count'] = count_mask_pixels(path)

        review_rows.append({**base, **outputs, **statuses})
        print('  Done.')

    except Exception as exc:
        msg = f'{type(exc).__name__}: {exc}'
        print('  ERROR:', msg)
        errors.append({**base, 'error': msg})

review_manifest = pd.DataFrame(review_rows)
error_log = pd.DataFrame(errors)

filter_label = f"{REVIEW_START_DATE or 'all'}_to_{REVIEW_END_DATE or 'all'}".replace(':', '').replace('/', '-')
manifest_path = TABLE_DIR / f'manual_review_manifest_lt{LOW_PROBA_THRESHOLD}_{filter_label}.csv'
error_path = TABLE_DIR / f'manual_review_errors_lt{LOW_PROBA_THRESHOLD}_{filter_label}.csv'

review_manifest.to_csv(manifest_path, index=False)
error_log.to_csv(error_path, index=False)

print('\nSaved review manifest:', manifest_path)
print('Saved error log:', error_path)
print('Rows in manifest:', len(review_manifest))
print('Errors:', len(error_log))

display(review_manifest.head(20))
if not error_log.empty:
    display(error_log)



[1/100] S2 2017-05-22 to 2017-05-23 | winam_s2_predictors_2017-05-22_to_2017-05-23
Staging for read: winam_s2_predictors_2017-05-22_to_2017-05-23_local_logistic_regression_baseline.tif
Staging for read: winam_s2_predictors_2017-05-22_to_2017-05-23_local_logistic_regression_baseline_proba.tif
Staging for read: winam_s2_predictors_2017-05-22_to_2017-05-23_local_rules.tif
  Done.

[2/100] S1 2025-08-30 to 2025-08-31 | winam_s1_scc_predictors_2025-08-30_to_2025-08-31
Staging for read: winam_s1_scc_predictors_2025-08-30_to_2025-08-31_local_logistic_regression_baseline.tif
Staging for read: winam_s1_scc_predictors_2025-08-30_to_2025-08-31_local_logistic_regression_baseline_proba.tif
Staging for read: winam_s1_scc_predictors_2025-08-30_to_2025-08-31_local_rules.tif
  Done.

[3/100] S2 2025-08-30 to 2025-08-31 | winam_s2_predictors_2025-08-30_to_2025-08-31
Staging for read: winam_s2_predictors_2025-08-30_to_2025-08-31_local_logistic_regression_baseline.tif
Staging for read: winam_s2_predictor

,sensor,start_date,end_date,prefix,class_tif,proba_tif,rule_tif,low_proba_threshold,lowconf_mask,floating_lowconf_mask,model_rule_disagreement_mask,lowconf_disagreement_mask,lowconf_mask_pixel_count,floating_lowconf_mask_pixel_count,model_rule_disagreement_mask_pixel_count,lowconf_disagreement_mask_pixel_count,lowconf_mask_status,floating_lowconf_mask_status,model_rule_disagreement_mask_status,lowconf_disagreement_mask_status
0,S2,2017-05-22,2017-05-23,winam_s2_predictors_2017-05-22_to_2017-05-23,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2017-05-22_to_2017-05-23_...,70,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2017-05-22_to_2017-05-23_lowconf_lt70.tif,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2017-05-22_to_2017-05-23_floating_lowconf_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2017-05-22_to_2017-05-23_model_rule_disagr...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2017-05-22_to_2017-05-23_lowconf_lt70_and_...,6793494,25196,2854425,2603494,written,written,written,written
1,S1,2025-08-30,2025-08-31,winam_s1_scc_predictors_2025-08-30_to_2025-08-31,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s1_scc_predictors_2025-08-30_to_2025-08...,70,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s1_scc_predictors_2025-08-30_to_2025-08-31_lowconf_lt70.tif,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s1_scc_predictors_2025-08-30_to_2025-08-31_floating_lowc...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s1_scc_predictors_2025-08-30_to_2025-08-31_model_rule_di...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s1_scc_predictors_2025-08-30_to_2025-08-31_lowconf_lt70_...,120202,30310,180676,48165,written,written,written,written
2,S2,2025-08-30,2025-08-31,winam_s2_predictors_2025-08-30_to_2025-08-31,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-08-30_to_2025-08-31_...,70,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2025-08-30_to_2025-08-31_lowconf_lt70.tif,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2025-08-30_to_2025-08-31_floating_lowconf_...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2025-08-30_to_2025-08-31_model_rule_disagr...,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/masks/winam_s2_predictors_2025-08-30_to_2025-08-31_lowconf_lt70_and_...,7256926,14280,772413,727612,written,written,written,written
3,S2,2025-09-02,2025-09-03,winam_s2_predictors_2025-09-02_to_2025-09-03,/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs/winam_s2_predictors_2025-09-02_to_2025-09-03_...,/content/drive/My

## 6b. Polygonise review masks → vector features for QGIS

The masks in section 6 are rasters: over the full Winam AOI a single low-confidence pixel is sub-pixel on screen when zoomed out, and a raster has no attribute table to navigate. This section turns each mask's connected pixel blobs into **vector polygons** (one feature per contiguous uncertain region), an **always-visible centroid points layer**, and optional **buffer-merged review zones** (nearby blobs dissolved into a few large navigation targets) — all written to one GeoPackage with attribute tables you can sort and zoom through.

**Layers written** (into `outputs/manual_review/vectors/review_polygons_*.gpkg`):

- `uncertain_polygons` — one feature per contiguous uncertain blob (exact extent).
- `uncertain_centroids` — one point per blob; stays visible at any zoom.
- `review_zones` — nearby blobs buffer-merged into larger zones, each tagged with how many blobs / pixels it aggregates. Use this for the shortest navigation list.

**Per-feature attributes:** `feat_id` (1 = largest), `n_pixels`, `area_m2`, `mask_type` (`lowconf` / `floating_lowconf` / `disagreement` / `lowconf_disagreement`), `sensor`, `start_date`, `end_date`, `cx`/`cy` (centroid lon/lat), `prefix`, `mask_tif`, plus blank `review_status` / `correct_class` / `reviewer` / `notes` you can edit directly in QGIS. Zones additionally carry `zone_id`, `n_blobs`, `total_pixels` and `zone_area_m2`.

**Settings:** `MIN_CLUSTER_PIXELS` drops blobs smaller than N pixels (raise it to hide single-pixel speckle); `POLY_CONNECTIVITY = 8` groups diagonally-touching pixels into one blob; `WRITE_CENTROID_POINTS` adds the points layer; `MAKE_REVIEW_ZONES` / `ZONE_BUFFER_M` control the merged zones (blobs whose edges are within ~2× the buffer merge into one zone).

**Using it in QGIS**

1. Add the GeoPackage `outputs/manual_review/vectors/review_polygons_*.gpkg` (it holds `uncertain_polygons`, `uncertain_centroids` and `review_zones`).
2. Right-click a layer → **Open Attribute Table**. Start with `review_zones` for the fewest, largest targets, or `uncertain_polygons` for blob-level detail.
3. Sort by `total_pixels` / `n_pixels` descending, or filter by `mask_type` / `sensor` / `start_date`.
4. Select a row, then **Zoom to Feature(s)** (magnifier button, or `Ctrl+J` for the selection) to jump straight to that region; load the matching `class_tif` / `proba_tif` and predictor composites to judge it.
5. The `uncertain_centroids` layer stays visible at any zoom — use it as an overview of where the uncertainty is concentrated.

This is a navigation aid built from the section-6 masks; digitise your actual correction **points** as described in section 9.

In [ ]:
# =====================================================================
# 6b. Polygonise review masks -> vector features for QGIS "Zoom to Feature"
#
# The low-confidence / disagreement masks are rasters, so single uncertain
# pixels vanish when the large AOI is zoomed out and there is no attribute
# table to navigate. This cell converts each mask's connected pixel blobs
# into vector POLYGONS (one feature per contiguous uncertain region), an
# always-visible CENTROID POINTS layer, and optional buffer-merged REVIEW
# ZONES (nearby blobs dissolved into a few large, clickable navigation
# targets) - all written to one GeoPackage. In QGIS: open the attribute
# table, sort by n_pixels (largest/most important first), select a row, and
# use "Zoom to Feature" (or "Flash Features").
# =====================================================================
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape as shapely_shape
from shapely.ops import unary_union

POLYGONISE_REVIEW_MASKS = True
POLY_CONNECTIVITY = 8           # 8 = group diagonally-touching pixels into one blob; 4 = stricter
MIN_CLUSTER_PIXELS = 1          # drop blobs smaller than this (raise it to ignore single-pixel speckle)
WRITE_CENTROID_POINTS = True    # points stay visible at any zoom, unlike sub-pixel polygons
MAKE_REVIEW_ZONES = True        # buffer+dissolve nearby blobs into fewer, larger review zones
ZONE_BUFFER_M = 50              # blobs whose edges are within ~2x this (m) merge into one zone
REVIEW_VECTOR_DIR = REVIEW_DIR / 'vectors'
REVIEW_VECTOR_GPKG = REVIEW_VECTOR_DIR / f'review_polygons_lt{LOW_PROBA_THRESHOLD}_{filter_label}.gpkg'

# Manifest mask column -> short label written to the 'mask_type' attribute.
REVIEW_MASK_COLUMNS = {
    'lowconf_mask': 'lowconf',
    'floating_lowconf_mask': 'floating_lowconf',
    'model_rule_disagreement_mask': 'disagreement',
    'lowconf_disagreement_mask': 'lowconf_disagreement',
}


def polygonise_review_mask(mask_path, connectivity=POLY_CONNECTIVITY, min_pixels=MIN_CLUSTER_PIXELS):
    """Return (list of {geometry, n_pixels}, crs) for the value==1 blobs in a uint8 mask."""
    mask_local = stage_for_read(mask_path)
    feats = []
    with rasterio.open(mask_local) as src:
        crs = src.crs
        transform = src.transform
        pixel_area = abs(transform.a * transform.e)
        arr = src.read(1)
        target = arr == 1
        if not target.any():
            return feats, crs
        for geom, value in rio_shapes(arr, mask=target, transform=transform, connectivity=connectivity):
            if value != 1:
                continue
            poly = shapely_shape(geom)
            n_pixels = int(round(poly.area / pixel_area)) if pixel_area else 0
            if n_pixels < min_pixels:
                continue
            feats.append({'geometry': poly, 'n_pixels': n_pixels})
    return feats, crs


def build_review_zones(blob_gdf, metric_crs, buffer_m=ZONE_BUFFER_M):
    """Buffer+dissolve nearby blobs into merged review zones, tagged with how
    many blobs / pixels each zone aggregates. Returns a GeoDataFrame in EPSG:4326."""
    metric = blob_gdf.to_crs(metric_crs)
    merged = unary_union(list(metric.geometry.buffer(buffer_m)))
    parts = list(merged.geoms) if merged.geom_type == 'MultiPolygon' else [merged]
    zones = gpd.GeoDataFrame({'geometry': parts}, crs=metric_crs).reset_index(drop=True)

    blob_pts = gpd.GeoDataFrame(
        blob_gdf[['n_pixels', 'mask_type', 'sensor', 'start_date', 'end_date']].reset_index(drop=True),
        geometry=metric.geometry.centroid.values, crs=metric_crs,
    )
    joined = gpd.sjoin(blob_pts, zones, predicate='within', how='left')
    agg = joined.groupby('index_right').agg(
        n_blobs=('n_pixels', 'size'),
        total_pixels=('n_pixels', 'sum'),
        sensors=('sensor', lambda s: ','.join(sorted({str(v) for v in s}))),
        mask_types=('mask_type', lambda s: ','.join(sorted({str(v) for v in s}))),
        start_date=('start_date', 'min'),
        end_date=('end_date', 'max'),
    )
    zones = zones.join(agg)
    zones['zone_area_m2'] = zones.geometry.area.round(1)
    zone_centroids_4326 = zones.geometry.centroid.to_crs('EPSG:4326')
    zones = zones.to_crs('EPSG:4326')
    zones['cx'] = zone_centroids_4326.x.values
    zones['cy'] = zone_centroids_4326.y.values
    zones = zones.sort_values('total_pixels', ascending=False).reset_index(drop=True)
    zones.insert(0, 'zone_id', np.arange(1, len(zones) + 1))
    for field in ['review_status', 'reviewer', 'notes']:
        zones[field] = ''
    zone_cols = ['zone_id', 'n_blobs', 'total_pixels', 'zone_area_m2', 'sensors', 'mask_types',
                 'start_date', 'end_date', 'cx', 'cy', 'review_status', 'reviewer', 'notes', 'geometry']
    return zones[[c for c in zone_cols if c in zones.columns]]


if not POLYGONISE_REVIEW_MASKS:
    print('POLYGONISE_REVIEW_MASKS is False; skipping vectorisation.')
elif not GEOPANDAS_AVAILABLE:
    print('geopandas is required to write vector layers. Uncomment the pip install in section 2 and rerun.')
elif 'review_manifest' not in globals() or review_manifest.empty:
    print('No review_manifest in memory. Run section 6 (mask creation) first.')
else:
    records = []
    mask_crs = None
    for _, row in review_manifest.iterrows():
        for col, mask_type in REVIEW_MASK_COLUMNS.items():
            mask_path = row.get(col, '')
            if not path_exists_nonempty(mask_path):
                continue
            feats, crs = polygonise_review_mask(mask_path)
            if mask_crs is None and crs is not None:
                mask_crs = crs
            for f in feats:
                records.append({
                    'geometry': f['geometry'],
                    'n_pixels': f['n_pixels'],
                    'mask_type': mask_type,
                    'sensor': row.get('sensor', ''),
                    'start_date': row.get('start_date', ''),
                    'end_date': row.get('end_date', ''),
                    'prefix': row.get('prefix', ''),
                    'mask_tif': str(mask_path),
                })

    if not records:
        print('No uncertain pixels found in any mask (nothing to polygonise).')
    else:
        gdf = gpd.GeoDataFrame(records, geometry='geometry', crs=mask_crs)

        # Metric projection for area + accurate centroids, then centroids back to lon/lat.
        metric_crs = None
        try:
            metric_crs = gdf.estimate_utm_crs()
            metric = gdf.to_crs(metric_crs)
            gdf['area_m2'] = metric.geometry.area.round(1)
            centroids_4326 = metric.geometry.centroid.to_crs('EPSG:4326')
        except Exception as exc:
            print('UTM reprojection unavailable; area/centroids fall back to layer CRS.',
                  type(exc).__name__, exc)
            metric_crs = None
            gdf['area_m2'] = np.nan
            centroids_4326 = gdf.geometry.centroid
        gdf['cx'] = centroids_4326.x.values
        gdf['cy'] = centroids_4326.y.values

        # Largest / most important uncertain regions first, then add editable review fields.
        gdf = gdf.sort_values('n_pixels', ascending=False).reset_index(drop=True)
        gdf.insert(0, 'feat_id', np.arange(1, len(gdf) + 1))
        for field in ['review_status', 'correct_class', 'reviewer', 'notes']:
            gdf[field] = ''

        column_order = ['feat_id', 'n_pixels', 'area_m2', 'mask_type', 'sensor',
                        'start_date', 'end_date', 'cx', 'cy', 'prefix', 'mask_tif',
                        'review_status', 'correct_class', 'reviewer', 'notes', 'geometry']
        gdf = gdf[[c for c in column_order if c in gdf.columns]]

        # Optional buffer-merged review zones (needs the metric projection).
        zones_gdf = None
        if MAKE_REVIEW_ZONES:
            if metric_crs is None:
                print('Skipping review zones: a metric (UTM) projection was not available.')
            else:
                zones_gdf = build_review_zones(gdf, metric_crs, ZONE_BUFFER_M)

        # Write everything to one GeoPackage (staged locally, then copied to Drive once).
        REVIEW_VECTOR_DIR.mkdir(parents=True, exist_ok=True)
        local_gpkg = (LOCAL_STAGE_DIR / REVIEW_VECTOR_GPKG.name) if USE_LOCAL_STAGING else REVIEW_VECTOR_GPKG
        if Path(local_gpkg).exists():
            Path(local_gpkg).unlink()
        gdf.to_file(local_gpkg, layer='uncertain_polygons', driver='GPKG', mode='w')

        if WRITE_CENTROID_POINTS:
            centroid_gdf = gpd.GeoDataFrame(
                gdf.drop(columns='geometry'),
                geometry=gpd.points_from_xy(gdf['cx'], gdf['cy']),
                crs='EPSG:4326',
            )
            centroid_gdf.to_file(local_gpkg, layer='uncertain_centroids', driver='GPKG', mode='a')

        if zones_gdf is not None:
            zones_gdf.to_file(local_gpkg, layer='review_zones', driver='GPKG', mode='a')

        if USE_LOCAL_STAGING:
            copy_local_to_final(local_gpkg, REVIEW_VECTOR_GPKG)

        # Per-(sensor, mask_type) feature-count summary for prioritising the review.
        summary = (gdf.groupby(['sensor', 'mask_type'])
                   .agg(features=('feat_id', 'size'),
                        total_pixels=('n_pixels', 'sum'),
                        largest_blob_px=('n_pixels', 'max'))
                   .reset_index())
        summary_path = TABLE_DIR / f'review_polygons_summary_lt{LOW_PROBA_THRESHOLD}_{filter_label}.csv'
        summary.to_csv(summary_path, index=False)

        layer_names = ['uncertain_polygons']
        if WRITE_CENTROID_POINTS:
            layer_names.append('uncertain_centroids')
        if zones_gdf is not None:
            layer_names.append('review_zones')
        print('Saved review vector GeoPackage:', REVIEW_VECTOR_GPKG)
        print('  layers:', ', '.join(layer_names))
        print('  uncertain blobs:', len(gdf),
              '| review zones:', (len(zones_gdf) if zones_gdf is not None else 'n/a'),
              f'(merged within ~{ZONE_BUFFER_M} m)' if zones_gdf is not None else '')
        print('Saved feature-count summary:', summary_path)
        display(summary)

## 7. Create a QGIS correction table template

Use this schema for your manually digitised point layer. In QGIS, create a new GeoPackage point layer and add these fields.

Minimum fields needed for feedback into the classifier:

- `sensor`
- `prefix`
- `dominant_class`
- `include`

Strongly recommended fields:

- `start_date`
- `end_date`
- `pred_class`
- `pred_proba`
- `rule_class`
- `review_confidence`
- `notes`


In [ ]:
correction_schema = pd.DataFrame([
    {'field': 'sensor', 'type': 'text', 'example': 'S2'},
    {'field': 'start_date', 'type': 'text/date', 'example': '2021-02-01'},
    {'field': 'end_date', 'type': 'text/date', 'example': '2021-03-01'},
    {'field': 'prefix', 'type': 'text', 'example': 'winam_s2_predictors_2021-02-01_to_2021-03-01'},
    {'field': 'pred_class', 'type': 'integer', 'example': 3},
    {'field': 'pred_proba', 'type': 'integer', 'example': 58},
    {'field': 'rule_class', 'type': 'integer', 'example': 2},
    {'field': 'dominant_class', 'type': 'text', 'example': 'F2'},
    {'field': 'review_confidence', 'type': 'text', 'example': 'high'},
    {'field': 'include', 'type': 'integer', 'example': 1},
    {'field': 'validation_only', 'type': 'integer', 'example': 0},
    {'field': 'reviewer', 'type': 'text', 'example': 'BMS'},
    {'field': 'notes', 'type': 'text', 'example': 'Coherent floating mat visible in false colour composite.'},
])

schema_path = TEMPLATE_DIR / 'manual_corrections_schema.csv'
correction_schema.to_csv(schema_path, index=False)
print('Saved schema CSV:', schema_path)
display(correction_schema)

# Also create an empty CSV template. A point layer itself should normally be created in QGIS as a GeoPackage.
empty_template_path = TEMPLATE_DIR / 'manual_corrections_empty_template.csv'
pd.DataFrame(columns=correction_schema['field'].tolist()).to_csv(empty_template_path, index=False)
print('Saved empty table template:', empty_template_path)


Saved schema CSV: /content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/templates/manual_corrections_schema.csv


,field,type,example
0,sensor,text,S2
1,start_date,text/date,2021-02-01
2,end_date,text/date,2021-03-01
3,prefix,text,winam_s2_predictors_2021-02-01_to_2021-03-01
4,pred_class,integer,3
5,pred_proba,integer,58
6,rule_class,integer,2
7,dominant_class,text,F2
8,review_confidence,text,high
9,include,integer,1


Saved empty table template: /content/drive/MyDrive/Winam_RF_Training_Data/outputs/manual_review/templates/manual_corrections_empty_template.csv


## 8. Optional: sample manually corrected QGIS points back into training rows

Use this after you have digitised correction points in QGIS.

Expected input: a GeoPackage point layer with the schema above.

The cell samples the **original predictor GeoTIFFs** listed in the run log, not the classified rasters. It then writes:

- `SV_S2_Training_manual_corrections_v1.csv`
- `SV_S1_Training_manual_corrections_v1.csv`
- `SV_S2_Training_augmented_v1.csv`
- `SV_S1_Training_augmented_v1.csv`

Set `RUN_POINT_SAMPLING = True` when ready.


In [ ]:
RUN_POINT_SAMPLING = False

# Path to the manually digitised point layer exported/saved from QGIS.
MANUAL_CORRECTIONS_GPKG = REVIEW_DIR / 'manual_corrections_v1.gpkg'
MANUAL_CORRECTIONS_LAYER = None   # None = first layer/default layer

# Original training CSVs from the classifier notebook.
S2_ORIGINAL_TRAINING_CSV = CLASSIFIER_ROOT / 'SV_S2_Training.csv'
S1_ORIGINAL_TRAINING_CSV = CLASSIFIER_ROOT / 'SV_S1_Training.csv'

# Output version label for correction and augmented training data.
CORRECTION_VERSION = 'v1'

S2_RAW_CLASS_MAPPING = {'W': 0, 'T': 1, 'F2': 2, 'A': 3}
S1_RAW_CLASS_MAPPING = {'W': 0, 'A': 0, 'T': 1, 'F1': 2}

S2_PREDICTORS_EXPECTED = [
    'AWEI_p95', 'AWEI', 'AWEInsh', 'NDMI', 'MNDWI', 'NDVI',
    'B', 'G', 'R', 'RE1', 'RE2', 'RE3', 'RE4', 'NIR', 'SWIR', 'SWIR2'
]
S1_PREDICTORS_EXPECTED = ['VH_p5', 'VH_corrected', 'VH_smooth']


In [ ]:
def parse_predictor_tifs(value):
    """Parse the run-log predictor_tifs field into a list of Path objects."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, (list, tuple)):
        return [Path(str(v)) for v in value if str(v).strip()]

    text = str(value).strip()
    if not text:
        return []

    # Try Python-list style strings first.
    if text.startswith('[') and text.endswith(']'):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple)):
                return [Path(str(v)) for v in parsed if str(v).strip()]
        except Exception:
            pass

    # Fall back to common separators used in CSV logs.
    for sep in ['|', ';']:
        if sep in text:
            return [Path(part.strip()) for part in text.split(sep) if part.strip()]

    return [Path(text)]


def get_predictor_paths_for_prefix(prefix):
    matches = run_log[run_log['prefix'].astype(str).eq(str(prefix))]
    if matches.empty:
        raise KeyError(f'No run-log row matched prefix: {prefix}')
    row = matches.iloc[0]
    paths = parse_predictor_tifs(row.get('predictor_tifs', ''))
    paths = [p for p in paths if p.exists()]
    if not paths:
        raise FileNotFoundError(f'No existing predictor GeoTIFF paths found for prefix: {prefix}')
    return paths


def raster_band_names(src, expected_names=None):
    names = list(src.descriptions) if src.descriptions else []
    names = [n if n not in [None, ''] else None for n in names]
    if expected_names is not None and (not names or any(n is None for n in names)):
        if len(expected_names) >= src.count:
            names = expected_names[:src.count]
    if not names or any(n is None for n in names):
        names = [f'band_{i}' for i in range(1, src.count + 1)]
    return names


def point_is_inside_raster(src, x, y):
    bounds = src.bounds
    return (bounds.left <= x <= bounds.right) and (bounds.bottom <= y <= bounds.top)


def sample_points_from_predictor_tifs(points_gdf, predictor_tifs, expected_names):
    """Sample predictor values for points. Handles tiled predictor GeoTIFFs by trying each tile."""
    out_rows = []

    # Keep source CRS for output, but reproject per raster if needed.
    for raster_path in predictor_tifs:
        raster_path = stage_for_read(raster_path)
        with rasterio.open(raster_path) as src:
            band_names = raster_band_names(src, expected_names)
            pts = points_gdf.to_crs(src.crs) if points_gdf.crs != src.crs else points_gdf.copy()

            for idx, geom in pts.geometry.items():
                if geom is None or geom.is_empty:
                    continue
                x, y = geom.x, geom.y
                if not point_is_inside_raster(src, x, y):
                    continue

                values = next(src.sample([(x, y)], indexes=list(range(1, src.count + 1))))
                values = values.astype('float64')

                # Treat all-nodata samples as unusable for this raster tile.
                nodata = src.nodata
                if nodata is not None and np.all(values == nodata):
                    continue
                if np.all(~np.isfinite(values)):
                    continue

                row = {'_manual_index': idx}
                for name, val in zip(band_names, values):
                    if nodata is not None and val == nodata:
                        row[name] = np.nan
                    else:
                        row[name] = float(val)
                out_rows.append(row)

    sampled = pd.DataFrame(out_rows)
    if sampled.empty:
        return sampled

    # If overlapping tiles sampled the same point, keep the first valid sample.
    sampled = sampled.drop_duplicates(subset=['_manual_index'], keep='first')
    return sampled


def geometry_to_geojson_point(gdf_4326):
    geo_values = []
    for geom in gdf_4326.geometry:
        if geom is None or geom.is_empty:
            geo_values.append(None)
        else:
            geo_values.append(json.dumps({'type': 'Point', 'coordinates': [float(geom.x), float(geom.y)]}))
    return geo_values


def make_training_rows_from_manual_points(manual_gdf):
    rows = []
    manual_gdf = manual_gdf.copy()

    if manual_gdf.crs is None:
        raise ValueError('Manual correction layer has no CRS. Set the CRS in QGIS before sampling.')

    # Clean required fields.
    manual_gdf['include'] = pd.to_numeric(manual_gdf.get('include', 1), errors='coerce').fillna(1).astype(int)
    manual_gdf = manual_gdf[manual_gdf['include'].eq(1)].copy()

    if manual_gdf.empty:
        raise RuntimeError('No manual correction points with include = 1.')

    required = ['sensor', 'prefix', 'dominant_class']
    missing = [c for c in required if c not in manual_gdf.columns]
    if missing:
        raise ValueError(f'Manual correction layer is missing required fields: {missing}')

    for (sensor, prefix), group in manual_gdf.groupby(['sensor', 'prefix']):
        sensor = str(sensor)
        prefix = str(prefix)
        expected = S2_PREDICTORS_EXPECTED if sensor == 'S2' else S1_PREDICTORS_EXPECTED
        mapping = S2_RAW_CLASS_MAPPING if sensor == 'S2' else S1_RAW_CLASS_MAPPING

        predictor_tifs = get_predictor_paths_for_prefix(prefix)
        print(f'Sampling {len(group)} {sensor} correction point(s) for prefix {prefix} from {len(predictor_tifs)} predictor GeoTIFF(s).')
        sampled = sample_points_from_predictor_tifs(group, predictor_tifs, expected)

        if sampled.empty:
            print('  Warning: no valid predictor samples found for this group.')
            continue

        group_attr = group.drop(columns='geometry').copy()
        group_attr['_manual_index'] = group_attr.index
        merged = group_attr.merge(sampled, on='_manual_index', how='inner')

        # Add geometry in EPSG:4326 as .geo.
        group_4326 = group.loc[merged['_manual_index']].to_crs('EPSG:4326')
        merged['.geo'] = geometry_to_geojson_point(group_4326)

        merged['class'] = merged['dominant_class'].map(mapping)
        merged['Location'] = merged.get('Location', 'Winam')
        if 'Date' not in merged.columns:
            if 'start_date' in merged.columns:
                merged['Date'] = merged['start_date']
            else:
                # Fallback: derive from run-log row.
                matched = run_log[run_log['prefix'].astype(str).eq(prefix)]
                merged['Date'] = matched['start_date'].iloc[0] if not matched.empty else ''
        merged['date'] = pd.to_datetime(merged['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
        merged['manual_correction_version'] = CORRECTION_VERSION
        merged['manual_source_prefix'] = prefix

        rows.append(merged)

    if not rows:
        raise RuntimeError('No valid manual correction samples were created.')

    out = pd.concat(rows, ignore_index=True)
    out = out.dropna(subset=['class', '.geo']).copy()
    out['class'] = out['class'].astype(int)
    return out


In [ ]:
if RUN_POINT_SAMPLING:
    if not GEOPANDAS_AVAILABLE:
        raise ImportError('geopandas is required for point sampling. Install geopandas/pyogrio and rerun.')
    if not Path(MANUAL_CORRECTIONS_GPKG).exists():
        raise FileNotFoundError(MANUAL_CORRECTIONS_GPKG)

    if MANUAL_CORRECTIONS_LAYER:
        manual_gdf = gpd.read_file(MANUAL_CORRECTIONS_GPKG, layer=MANUAL_CORRECTIONS_LAYER)
    else:
        manual_gdf = gpd.read_file(MANUAL_CORRECTIONS_GPKG)

    print('Loaded manual correction points:', len(manual_gdf))
    print('Columns:', list(manual_gdf.columns))

    correction_rows = make_training_rows_from_manual_points(manual_gdf)
    print('Created correction training rows:', len(correction_rows))
    display(correction_rows.head())

    # Split by sensor and write correction-only and augmented CSVs.
    for sensor, original_csv, expected_predictors in [
        ('S2', S2_ORIGINAL_TRAINING_CSV, S2_PREDICTORS_EXPECTED),
        ('S1', S1_ORIGINAL_TRAINING_CSV, S1_PREDICTORS_EXPECTED),
    ]:
        sub = correction_rows[correction_rows['sensor'].astype(str).eq(sensor)].copy()
        if sub.empty:
            print(f'No {sensor} correction rows; skipping.')
            continue

        # Keep columns likely to be consumed by the classifier notebook.
        preferred_cols = ['.geo', 'dominant_class', 'class', 'Location', 'Date', 'date']
        meta_cols = ['sensor', 'prefix', 'manual_correction_version', 'manual_source_prefix',
                     'review_confidence', 'validation_only', 'reviewer', 'notes']
        available_cols = [c for c in preferred_cols + expected_predictors + meta_cols if c in sub.columns]
        sub_out = sub[available_cols].copy()

        correction_csv = CLASSIFIER_ROOT / f'SV_{sensor}_Training_manual_corrections_{CORRECTION_VERSION}.csv'
        sub_out.to_csv(correction_csv, index=False)
        print(f'Saved {sensor} correction-only CSV:', correction_csv)

        if Path(original_csv).exists():
            original = pd.read_csv(original_csv)
            # Preserve original column order, but include any new manual/meta columns after.
            for col in sub_out.columns:
                if col not in original.columns:
                    original[col] = np.nan
            for col in original.columns:
                if col not in sub_out.columns:
                    sub_out[col] = np.nan
            sub_out = sub_out[original.columns]
            augmented = pd.concat([original, sub_out], ignore_index=True)
            augmented_csv = CLASSIFIER_ROOT / f'SV_{sensor}_Training_augmented_{CORRECTION_VERSION}.csv'
            augmented.to_csv(augmented_csv, index=False)
            print(f'Saved {sensor} augmented training CSV:', augmented_csv)
        else:
            print(f'Original {sensor} CSV not found, so augmented CSV was not written:', original_csv)
else:
    print('RUN_POINT_SAMPLING is False. Set it to True after digitising manual correction points in QGIS.')


RUN_POINT_SAMPLING is False. Set it to True after digitising manual correction points in QGIS.


## 9. How to use the outputs in QGIS

Load the following layers for each scene you want to review:

1. `class_tif` from the manifest.
2. `proba_tif` from the manifest.
3. `rule_tif` from the manifest, where available.
4. `lowconf_mask`, `floating_lowconf_mask`, and/or `lowconf_disagreement_mask` from the manifest.
5. The original predictor GeoTIFFs from the run log for visual composites.

Digitise correction **points**, not whole raster regions. Add points in homogeneous areas and fill in at least:

```text
sensor
prefix
dominant_class
include
```

Use `include = 1` for points that should become training samples. Use `include = 0` for ambiguous points you want to keep for audit only.

After saving the point layer, return to section 8, set `RUN_POINT_SAMPLING = True`, and run the sampling cells.
